# A02 – Data Source Setup
Kiểm tra 9 CSV Olist, profiling dữ liệu nguồn và tạo ngày lễ quốc gia Brazil 2016–2018. Notebook chạy từ một bản clone repository và không dùng Google Drive.

In [ ]:
from pathlib import Path
import sys

IN_COLAB = 'google.colab' in sys.modules
print('Google Colab:', IN_COLAB)

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / '.git').exists() and (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError('Clone/open the CommerceFlow360 repository first.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

## Cài dependencies
Bỏ comment và chạy cell dưới nếu môi trường chưa có dependencies. Trên Colab, lệnh đọc trực tiếp `requirements.txt` từ repository vừa clone.

In [ ]:
# %pip install -q -r {PROJECT_ROOT / 'requirements.txt'}

In [ ]:
OLIST_FILES = [
    'olist_customers_dataset.csv',
    'olist_geolocation_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
    'product_category_name_translation.csv',
]
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
available = [name for name in OLIST_FILES if (RAW_DIR / name).is_file()]
missing = [name for name in OLIST_FILES if name not in available]
print(f'Found {len(available)}/9 Olist CSV files')
print('Available:', available)
print('Missing:', missing)

## Tùy chọn: tải riêng file còn thiếu từ Kaggle
Cell không chạy mặc định và không ghi đè file sẵn có. Nếu Kaggle yêu cầu xác thực, cấu hình credential bằng Colab Secrets/biến môi trường; không dán token vào notebook.

In [ ]:
DOWNLOAD_MISSING = False
if DOWNLOAD_MISSING and missing:
    import shutil
    import kagglehub
    download_dir = Path(kagglehub.dataset_download('olistbr/brazilian-ecommerce'))
    for name in missing:
        candidates = list(download_dir.rglob(name))
        if not candidates:
            print('Not found in Kaggle download:', name)
            continue
        destination = RAW_DIR / name
        if not destination.exists():
            shutil.copy2(candidates[0], destination)
            print('Copied:', name)
elif missing:
    print('Set DOWNLOAD_MISSING=True to download only missing files.')

In [ ]:
import pandas as pd
from IPython.display import display

inventory = []
for name in OLIST_FILES:
    path = RAW_DIR / name
    if path.is_file():
        header = pd.read_csv(path, nrows=0)
        inventory.append({'file_name': name, 'column_count': len(header.columns), 'size_mb': round(path.stat().st_size / 1024**2, 3)})
display(pd.DataFrame(inventory))

In [ ]:
from src.validation.profile_raw_data import run as run_profiling
profile_result = run_profiling(PROJECT_ROOT)
display(profile_result)

In [ ]:
from src.ingestion.generate_brazil_holidays import generate
holiday_result = generate(PROJECT_ROOT)
display(holiday_result)

In [ ]:
checks = {
    '9 Olist CSV files': all((RAW_DIR / name).is_file() for name in OLIST_FILES),
    'all Olist schemas pass': len(profile_result) == 9 and profile_result['status'].eq('PASS').all(),
    'comparison output exists': (PROJECT_ROOT / 'data/profiling/doi_chieu_so_dong.csv').is_file(),
    'holiday output exists': (RAW_DIR / 'holidays_brazil.csv').is_file(),
    'holiday dates unique': not holiday_result['date'].duplicated().any(),
    'holiday required fields complete': not holiday_result.isna().any().any(),
    'holiday years are 2016-2018': set(pd.to_datetime(holiday_result['date']).dt.year) == {2016, 2017, 2018},
}
check_df = pd.DataFrame([{'check': name, 'status': 'PASS' if passed else 'FAIL'} for name, passed in checks.items()])
display(check_df)
assert all(checks.values()), 'A02 validation failed; inspect the table above.'
print('FINAL STATUS: PASS')